# Deep dive analysis

In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pyspark.sql.functions as F
import pandas as pd

lst_mthend='2025-09-30'
exrt_string = f'''
with xrt as (
select  cast(XCHNG_RATE as int) ex_rate,
        row_number() over (partition by XCHNG_RATE_TYP order by FR_EFF_DT DESC) rn
from    vn_published_cas_db.texchange_rates
where   XCHNG_RATE_TYP='U'
    and FR_CRCY_CODE='78'
    and to_date(FR_EFF_DT) <= '{lst_mthend}'
qualify rn=1
) select ex_rate from xrt
'''
exrt_df = sql_to_df(exrt_string, 1, spark)
ex_rate = exrt_df.collect()[0][0]
del exrt_df
print(ex_rate)

In [0]:
df_cols = [
    "POL_NUM", "CLI_NUM", "CHANNEL", "SUBMIT_TYPE", "BENEFIT_RIDER", "AIS_DECISION", "FINAL_UW_DECISION",
    "STP_FLAG", "BASE_APE_cat", "HIT_RULE", "ASSIGN_TEAM", "AIS_RULE_CATEGORY", "AIS_RULE_NAMES",
    "ADMIN_RULE_CATEGORY", "ADMIN_REMARKS", "DIGITEXX_REMARK_CATEGORY", "DIGITEXX_REMARKS",
    "TAT_SBMT_FUD", "TAT_FUD_ISSUE", "TAT_SBMT_ISSUE", "EXISTING_PO_IND", "EXISTING_LA_IND",
    "SBMT_DT", "ISSUE_DATE", "RIDER_APE_cat", "TOTAL_CLAIM_AMOUNT_cat", 'CLAIM_AMOUNT_PER_YEAR_cat', 
    'CLAIM_BENEFITS', 'CLAIM_MONTHS', 'MC_CLAIMS', 'MC_CLAIM_AMOUNT', 'HCR_CLAIMS', 'HCR_CLAIM_AMOUNT', 
    'FEMALE_CLAIMS', 'FEMALE_CLAIM_AMOUNT', 'OTHER_CLAIMS', 'OTHER_CLAIM_AMOUNT', 'VALID_HIS_CLM_IND',
    "AGENT_TIER", "AGE_LA_cat"
]

# Few columns for existing sales analysis
df_cols_2 = [
    "pol_num", "cli_num", "channel", "stp_flag", "ais_decision", "existing_po_ind", "existing_la_ind"
]

df_path = "/mnt/lab/vn/project/VN_NB/NB_UW/"

df = spark.read.format("parquet").load(df_path).filter(
    (F.col("POLICY_STATUS") == "APPROVED")
    &(F.col("CHANNEL") == "AGENCY")
    &(F.col("SUBMIT_TYPE") != "SIO")
    #&(~F.col("PLAN_CODE").isin(['PN001','PA007','PA008','FDB01','BIC01','BIC02','BIC03','BIC04','CA360','CI360','CX360','MI007']))
)#.select(*[col.lower() for col in df_cols])

df = df.withColumn(
    "existing_ind",
    F.greatest(F.col("EXISTING_PO_IND"), F.col("EXISTING_LA_IND"))
)

# print("Final :", df.count(), df.select("POL_NUM").distinct().count())
# df.createOrReplaceTempView("nb_pol_list")

In [0]:
# df.filter(
#     F.col("EXISTING_PO_IND") == "Y"
# ).groupBy(
#     "CHANNEL"
# ).agg(
#     F.countDistinct("PO_NUM").alias("po_count"),
#     F.countDistinct("CLI_NUM").alias("cli_count")
# ).orderBy(
#     F.desc("CHANNEL")
# ).display()

In [0]:
df.toPandas().to_csv("/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/VN_NB_analysis_rawdata_v2.csv", index=False, header=True, encoding='utf-8-sig')

# Analyis section

In [0]:
df = df.filter(F.col("ais_decision").isNotNull())

df.groupBy(
    "stp_flag"
).agg(
    F.count(F.col("pol_num")).alias("pol_count")
).display()

In [0]:
df.filter(
    (F.col("STP_FLAG") == False)
    &(F.col("HIT_RULE") == 1)
).groupBy(
    "AIS_DECISION"
).agg(
    F.count(F.col("POL_NUM")).alias("POL_COUNT")
).orderBy(
    "AIS_DECISION"
).display()

In [0]:
df.filter(
     (F.col("STP_FLAG") == False)
    &(F.col("AIS_DECISION") != "Clean")
    #&(F.col("HIT_RULE") == 1)
).groupBy(
    "EXISTING_PO_IND",
    "BASE_APE_cat"
).agg(
    F.count(F.col("POL_NUM")).alias("POL_COUNT"),
    F.min(F.col("TAT_SBMT_ISSUE")).alias("SHORTEST_TAT"),
    F.max(F.col("TAT_SBMT_ISSUE")).alias("LONGEST_TAT"),
    F.avg(F.col("TAT_SBMT_ISSUE")).alias("AVG_TAT")
).orderBy(
    F.desc("EXISTING_PO_IND"),
    "BASE_APE_cat"
).display()

In [0]:
df.filter(
     (F.col("STP_FLAG") == False)
    &(F.col("AIS_DECISION") != "Clean")
    &(F.col("HIT_RULE") == 1)
).groupBy(
    #"STP_FLAG",
    #"AIS_DECISION",
    "ASSIGN_TEAM"
).agg(
    F.count(F.col("POL_NUM")).alias("POL_COUNT")
).display()

In [0]:
df.groupBy(
    "STP_FLAG",
    "HIT_RULE",
    "AIS_DECISION",
    "ASSIGN_TEAM",
    "FINAL_UW_DECISION",
    "AIS_RULE_CATEGORY",
    "DIGITEXX_REMARK_CATEGORY"
).agg(
    F.count("POL_NUM").alias("POL_COUNT"),
    F.min(F.col("TAT_SBMT_ISSUE")).alias("SHORTEST_TAT"),
    F.max(F.col("TAT_SBMT_ISSUE")).alias("LONGEST_TAT"),
    F.avg(F.col("TAT_SBMT_ISSUE")).alias("AVG_TAT")
).orderBy(
    F.desc("POL_COUNT")
).display()

In [0]:
df.filter(
     (F.col("STP_FLAG") == False)
    &(F.col("AIS_DECISION") != F.lit("Clean"))
    &(F.col("ASSIGN_TEAM").isNotNull())
    &(F.col("HIT_RULE") == 1)
    &(F.col("ASSIGN_TEAM").like("%Admin%"))
).groupBy(
    "ASSIGN_TEAM",
    #"AIS_RULE_CATEGORY",
    "ADMIN_RULE_CATEGORY"
).agg(
    F.count("POL_NUM").alias("POL_COUNT"),
    F.min(F.col("TAT_SBMT_ISSUE")).alias("SHORTEST_TAT"),
    F.max(F.col("TAT_SBMT_ISSUE")).alias("LONGEST_TAT"),
    F.avg(F.col("TAT_SBMT_ISSUE")).alias("AVG_TAT")
).orderBy(
    F.desc("POL_COUNT")
).display()

In [0]:
df.filter(
     (F.col("STP_FLAG") == False)
    &(F.col("AIS_DECISION") != F.lit("Clean"))
    &(F.col("ASSIGN_TEAM").isNotNull())
    &(F.col("HIT_RULE") == 1)
    &(F.col("ASSIGN_TEAM").like("%UW%"))
    &(F.col("AIS_RULE_CATEGORY").like("%Policy and Administrative Checks%"))
).groupBy(
    "ASSIGN_TEAM",
    "AIS_RULE_CATEGORY",
    "AIS_RULE_NAMES"
).agg(
    F.count("POL_NUM").alias("POL_COUNT"),
    F.min(F.col("TAT_SBMT_ISSUE")).alias("SHORTEST_TAT"),
    F.max(F.col("TAT_SBMT_ISSUE")).alias("LONGEST_TAT"),
    F.avg(F.col("TAT_SBMT_ISSUE")).alias("AVG_TAT")
).orderBy(
    F.desc("POL_COUNT")
).display()

In [0]:
ais_admin_rules = df.filter(
     (F.col("STP_FLAG") == False)
    &(F.col("AIS_DECISION") != F.lit("Clean"))
    &(F.col("ASSIGN_TEAM").isNotNull())
    &(F.col("HIT_RULE") == 1)
    &(F.col("ASSIGN_TEAM").like("%UW%"))
    &(F.col("AIS_RULE_CATEGORY").like("%Policy and Administrative Checks%"))
    &(F.col("AIS_RULE_NAMES").like("%NB underwriting check%"))
).withColumn(
    "AIS_RULE_NAMES",
    F.lit("NB underwriting check")
)

ais_admin_rules.groupBy(
    "AIS_RULE_NAMES",
    "ADMIN_RULE_CATEGORY",
).agg(
    F.count("POL_NUM").alias("POL_COUNT"),
    F.min(F.col("TAT_SBMT_ISSUE")).alias("SHORTEST_TAT"),
    F.max(F.col("TAT_SBMT_ISSUE")).alias("LONGEST_TAT"),
    F.avg(F.col("TAT_SBMT_ISSUE")).alias("AVG_TAT")
).orderBy(
    F.desc("POL_COUNT")
).display()

In [0]:
ais_admin_rules_2 = df.filter(
     (F.col("STP_FLAG") == False)
    &(F.col("AIS_DECISION") != F.lit("Clean"))
    &(F.col("ASSIGN_TEAM").isNotNull())
    &(F.col("HIT_RULE") == 1)
    &(F.col("ASSIGN_TEAM").like("%UW%"))
    &(F.col("AIS_RULE_CATEGORY").like("%Policy and Administrative Checks%"))
    &(F.col("AIS_RULE_NAMES").like("%NB underwriting check%"))
    &(F.col("EXISTING_PO_IND") == "N")
    &(F.col("EXISTING_LA_IND") == "N")
).withColumn(
    "AIS_RULE_NAMES",
    F.lit("NB underwriting check")
)

ais_admin_rules_2.groupBy(
    "BASE_APE_cat",
    "ADMIN_RULE_CATEGORY",
).agg(
    F.count("POL_NUM").alias("POL_COUNT"),
    F.min(F.col("TAT_SBMT_ISSUE")).alias("SHORTEST_TAT"),
    F.max(F.col("TAT_SBMT_ISSUE")).alias("LONGEST_TAT"),
    F.avg(F.col("TAT_SBMT_ISSUE")).alias("AVG_TAT")
).orderBy(
    F.desc("POL_COUNT")
).display()

1)	Existing customer buying policy

In [0]:
existing_po_pur_hist = spark.sql("""
with fld_nm as (
select  fld_valu as OWN_INS_REL,
        case 
          when FLD_VALU_DESC_ENG = 'Self' then    '1. Self'
          when FLD_VALU_DESC_ENG = 'parent' then  '2. Child'
          when FLD_VALU_DESC_ENG = 'Spouse' then  '3. Spouse'
          when FLD_VALU_DESC_ENG = 'Child' then   '4. Parent'
          else '5. Relative'
        end as BOUGHT_FOR
from    vn_published_cas_db.tfield_values
where   FLD_NM = 'REL_TO_INSRD'
), cvg_ape as (
select  pol.pol_num, cast(sum(cvg_prem*12/cast(pol.pmt_mode as int)) as int) as cvg_ape
from    vn_published_casm_cas_snapshot_db.tcoverages cvg
   inner join
        vn_published_casm_cas_snapshot_db.tpolicys pol on cvg.pol_num = pol.pol_num and cvg.image_date = pol.image_date
where   cvg.image_date = '2025-03-31'
   and  cvg.cvg_prem > 0
group by pol.pol_num
), rider_type as (
select  pol_num, 
        concat_ws('/', 
            collect_set(case when cvg.cvg_typ='R' then PROD_TYP end)) as rider_type
from    vn_published_casm_cas_snapshot_db.tcoverages cvg
inner join
        vn_published_cas_db.tplans pln on cvg.plan_code=pln.plan_code and cvg.vers_num=pln.vers_num
where   cvg.image_date = '2025-03-31'
group by
        pol_num
), hcr as (
select  pol_num,
        concat_ws('/',
            collect_set(fld.bought_for)) as bought_hcr_for,
        max(substr(cvg.CVG_CLAS,1,1)) as hcr_class 
from    vn_published_casm_cas_snapshot_db.tcoverages cvg
inner join
        vn_published_cas_db.tplans pln on cvg.plan_code=pln.plan_code and cvg.vers_num=pln.vers_num
left join
        fld_nm fld on cvg.REL_TO_INSRD = fld.OWN_INS_REL
where   cvg.image_date = '2025-03-31'
    and cvg.plan_code like 'RHC%'
group by
        pol_num
), mc as (
select  pol_num,
        concat_ws('/',
            collect_set(fld.bought_for)) as bought_mc_for,
        max(substr(cvg.CVG_CLAS,1,1)) as mc_class 
from    vn_published_casm_cas_snapshot_db.tcoverages cvg
inner join
        vn_published_cas_db.tplans pln on cvg.plan_code=pln.plan_code and cvg.vers_num=pln.vers_num
left join
        fld_nm fld on cvg.REL_TO_INSRD = fld.OWN_INS_REL
where   cvg.image_date = '2025-03-31'
    and cvg.plan_code like 'MC%'
group by
        pol_num
), frst_bought_by_existing_po as (
select  cpl.cli_num as po_num,
        min(sbmt_dt) as frst_sbmt_dt_by_existing_po
from    {nb_pol_list} nb
inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on 
        (nb.pol_num = cpl.pol_num and cpl.link_typ = 'O' and cpl.rec_status = 'A')
where   existing_po_ind = 'Y'
  and   cpl.image_date = '2025-03-31'
group by
        cpl.cli_num
), last_holding as (
select  cpl.cli_num as po_num,
        cast(
           sum(pol.mode_prem * 12 / (cast(pol.pmt_mode as int)))
        as int) as yearly_prem_before,
        count(distinct pol.pol_num) as yearly_pol_before
from    vn_published_casm_cas_snapshot_db.tpolicys pol
inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on 
        (pol.pol_num = cpl.pol_num and pol.image_date = cpl.image_date and cpl.link_typ = 'O' and cpl.rec_status = 'A')
inner join
        frst_bought_by_existing_po nb on (cpl.cli_num = nb.po_num and pol.pol_eff_dt < nb.frst_sbmt_dt_by_existing_po)
where   pol.image_date = '2025-03-31'
    and pol.pol_stat_cd not in ('A','N','R','X')
    and pol.mode_prem > 0
group by cpl.cli_num
), last_pur_pols as (
select  nb.po_num, months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) as months_since_last_purchase,
        case 
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 2 then  '1. < 2m'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 4 then  '2. 2 - 4m'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 6 then  '3. 4 - 6m'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 12 then '4. 6m - 1yr'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 24 then '5. 1 - 2yr'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 36 then '6. 2 - 3yr'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 60 then '7. 3 - 5yr'
          else '8. >5yr'
        end as mth_since_lst_pur_cat,
        pol.pol_num, to_date(pol.pol_eff_dt) as pol_eff_dt,
        case when pol.plan_code_base in ('UL004','UL005','UL035','UL036') then 'UL-Single-Premium'
              when pol.plan_code_base not in ('UL004','UL005','UL035','UL036') and pol.plan_code_base like 'UL%' then 'ULI'
              when pol.plan_code_base like 'RUV%' then 'UILP'
              when pol.plan_code_base like 'RP%' then 'TROP'
              when pol.plan_code_base in ('BHC9I','PN001','PA007','PA008') then 'Single-Year'
              when pol.plan_code_base in ('FDB01','BIC01','BIC02','BIC03','BIC04') then 'Digital'
              when pol.plan_code_base in ('CA360','CI360','CN360','CX360','MI007') then 'Activator-Micro'
              else 'Traditional'
        end as product_type,
        nvl(cvg.cvg_ape,0) as total_ape,
        case
            when nvl(cvg.cvg_ape,0) = 0 then '_Matured/Expired_' 
            when cvg.cvg_ape <  10000 then 'a. <10M'
            when cvg.cvg_ape <= 15000 then 'b. 10 - 15M'
            when cvg.cvg_ape <= 20000 then 'c. 15 - 20M'
            when cvg.cvg_ape <= 25000 then 'd. 20 - 25M'
            when cvg.cvg_ape <= 30000 then 'e. 25 - 30M'
            when cvg.cvg_ape <= 50000 then 'f. 30 - 50M'
            when cvg.cvg_ape >  50000 then 'g. >50M'
        end as total_ape_cat,
        case
            when rid.rider_type is null then 'h. None' 
            when rid.rider_type = 'HC_RIDER' then 'a. HCR Only'
            when rid.rider_type = 'MC_RIDER' then 'b. MC Only'
            when rid.rider_type = 'CI_RIDER' then 'c. CI Only'
            when rid.rider_type like '%HC_RIDER%' and rid.rider_type like '%/%'
                                             then 'd. Multiple w/ HCR'
            when rid.rider_type like '%MC_RIDER%' and rid.rider_type like '%/%'
                                             then 'e. Multiple w/ MC'
            when rid.rider_type like '%CI_RIDER%' and rid.rider_type like '%/%'
                                             then 'f. Multiple w/ CI'
            else 'g. Others'
        end as rider_type,
        hcr.hcr_class,
        hcr.bought_hcr_for,
        mc.bought_mc_for,
        fld.bought_for,
        row_number() over (partition by nb.po_num order by pol.pol_eff_dt desc, pol.pol_num desc) rn
from    vn_published_casm_cas_snapshot_db.tpolicys pol
inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on 
        (pol.pol_num = cpl.pol_num and pol.image_date = cpl.image_date and cpl.link_typ = 'O' and cpl.rec_status = 'A')
inner join
        frst_bought_by_existing_po nb on (cpl.cli_num = nb.po_num and pol.pol_eff_dt <= nb.frst_sbmt_dt_by_existing_po)
left join
        cvg_ape cvg on (pol.pol_num = cvg.pol_num)
left join
        rider_type rid on (pol.pol_num = rid.pol_num)
left join
        hcr on (pol.pol_num = hcr.pol_num)
left join
        mc on (pol.pol_num = mc.pol_num)
left join
        fld_nm fld on pol.OWN_INS_REL = fld.OWN_INS_REL
where   1 = 1
  and   pol.pol_stat_cd not in ('A', 'N', 'R', 'X')
  and   pol.image_date = '2025-03-31'
qualify rn = 1
), post_year_holding as (
select  cpl.cli_num as po_num,
        cast(sum(pol.mode_prem*12/cast(pol.pmt_mode as int)) as int) as total_ape_post_year,
        count(distinct pol.pol_num) as total_policy_post_year
from    vn_published_casm_cas_snapshot_db.tpolicys pol
inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on 
        (pol.pol_num = cpl.pol_num and pol.image_date = cpl.image_date and cpl.link_typ = 'O' and cpl.rec_status = 'A')
inner join
        frst_bought_by_existing_po nb on (cpl.cli_num = nb.po_num and months_between(pol.sbmt_dt, nb.frst_sbmt_dt_by_existing_po) >= 12)
where   pol.image_date = '2025-03-31'
   and  pol.pol_stat_cd not in ('A','N','R','X')
   and  pol.mode_prem > 0
group by cpl.cli_num
), po_cli as (
select  cpl.cli_num as po_num, nb.cli_num,
        case when nb.PLAN_CODE in ('UL004','UL005','UL035','UL036') then 'UL-Single-Premium'
              when nb.PLAN_CODE not in ('UL004','UL005','UL035','UL036') and nb.PLAN_CODE like 'UL%' then 'ULI'
              when nb.PLAN_CODE like 'RUV%' then 'UILP'
              when nb.PLAN_CODE like 'RP%' then 'TROP'
              when nb.PLAN_CODE in ('BHC9I','PN001','PA007','PA008') then 'Single-Year'
              when nb.PLAN_CODE in ('FDB01','BIC01','BIC02','BIC03','BIC04') then 'Digital'
              when nb.PLAN_CODE in ('CA360','CI360','CN360','CX360','MI007') then 'Activator-Micro'
              else 'Traditional'
        end as product_type,
        nb.tot_ape as total_ape,
        case 
            when nb.tot_ape <  10000 then 'a. <10M'
            when nb.tot_ape <= 15000 then 'b. 10 - 15M'
            when nb.tot_ape <= 20000 then 'c. 15 - 20M'
            when nb.tot_ape <= 25000 then 'd. 20 - 25M'
            when nb.tot_ape <= 30000 then 'e. 25 - 30M'
            when nb.tot_ape <= 50000 then 'f. 30 - 50M'
            when nb.tot_ape >  50000 then 'g. >50M'
        end as total_ape_cat,
        case
            when nb.benefit_rider is null or nb.benefit_rider = 'NONE' 
                                             then 'h. None' 
            when nb.benefit_rider = 'HC_RIDER' then 'a. HCR Only'
            when nb.benefit_rider = 'MC_RIDER' then 'b. MC Only'
            when nb.benefit_rider = 'CI_RIDER' then 'c. CI Only'
            when nb.benefit_rider like '%HC_RIDER%' and nb.benefit_rider like '%/%'
                                             then 'd. Multiple w/ HCR'
            when nb.benefit_rider like '%MC_RIDER%' and nb.benefit_rider like '%/%'
                                             then 'e. Multiple w/ MC'
            when nb.benefit_rider like '%CI_RIDER%' and nb.benefit_rider like '%/%'
                                             then 'f. Multiple w/ CI'
            else 'g. Others'
        end as rider_type,
        nvl(hcr.hcr_class,'None') as hcr_class,
        hcr.bought_hcr_for,
        mc.bought_mc_for,
        existing_po_ind,
        bought_for
from    {nb_pol_list} nb
inner join
        vn_published_casm_cas_snapshot_db.tpolicys pol on nb.pol_num = pol.pol_num
inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on 
        (pol.pol_num = cpl.pol_num and pol.image_date = cpl.image_date and cpl.link_typ = 'O' and cpl.rec_status = 'A')
left join
        rider_type rid on pol.pol_num = rid.pol_num
left join
        hcr on (pol.pol_num = hcr.pol_num)
left join
        mc on (pol.pol_num = mc.pol_num)
left join
        fld_nm fld on pol.OWN_INS_REL = fld.OWN_INS_REL
where   existing_po_ind = 'Y'
  and   pol.image_date = '2025-03-31'
)
select  at.po_num, at.cli_num, 
        case when bf.months_since_last_purchase < 24 then
            case when bf.months_since_last_purchase < 9 then
                        concat('MOB0', 
                                lpad(bf.months_since_last_purchase + 1, 1, '0'))
                   else concat('MOB', lpad(bf.months_since_last_purchase + 1, 2, '0'))
                end
            else 'MOB24+'
        end as mob, 
        lh.yearly_prem_before, lh.yearly_pol_before,
        py.total_ape_post_year, py.total_policy_post_year,
        bf.total_ape as total_ape_before, at.total_ape as total_ape_after,
        cast(case when bf.total_ape > 0 then ((at.total_ape - bf.total_ape) / bf.total_ape)
                else null
        end as decimal(5,2)) as ape_gap_pct,
        case when nvl(bf.total_ape,0) == 0      then 'Renew'
             when at.total_ape > bf.total_ape   then 'Up'
             when at.total_ape == bf.total_ape  then 'Same'
             when at.total_ape < bf.total_ape   then 'Down'
        end as ape_movement_cat,   
        bf.bought_for as bought_for_before, at.bought_for as bought_for_after, 
        bf.product_type as product_bought_before, at.product_type as product_bought_after,
        bf.total_ape_cat as ape_cat_before, at.total_ape_cat as ape_cat_after,
        nvl(bf.rider_type,'g. None') as rider_type_before, at.rider_type as rider_type_after,
        bf.bought_hcr_for as bought_hcr_for_before, at.bought_hcr_for as bought_hcr_for_after,
        bf.bought_mc_for as bought_mc_for_before, at.bought_mc_for as bought_mc_for_after,
        nvl(bf.hcr_class,'None') as rider_hcr_before, at.hcr_class as rider_class_after,
        bf.mth_since_lst_pur_cat
        --, count(distinct at.po_num)  as number_po, 
        --count(distinct at.cli_num) as number_cli
from    po_cli at 
   left join
        last_pur_pols bf on at.po_num = bf.po_num
   left join
        last_holding lh on at.po_num = lh.po_num
   left join
        post_year_holding py on at.po_num = py.po_num
where   bf.bought_for is not null
/*group by bf.bought_for, at.bought_for, bf.product_type, at.product_type, 
        bf.total_ape_cat, at.total_ape_cat, 
        nvl(bf.rider_type,'g. None'), at.rider_type,
        at.bought_hcr_for,
        nvl(bf.hcr_class,'None'), at.hcr_class,
        bf.mth_since_lst_pur_cat*/
""", 
nb_pol_list = df)
existing_po_pd = existing_po_pur_hist.toPandas()
existing_po_pd.to_csv(
    f"/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/existing_po_pur_hist_details.csv",
    index=False,
    header=True
)
print(existing_po_pd.shape[0])
existing_po_pd.head(2)

existing_po_pur_sum = spark.sql("""
select  bought_for_before, bought_for_after,
        product_bought_before, product_bought_after,
        ape_cat_before, ape_cat_after,
        rider_type_before, rider_type_after,
        bought_hcr_for_before, bought_hcr_for_after, rider_hcr_before, rider_class_after,
        bought_mc_for_before, bought_mc_for_after,
        mth_since_lst_pur_cat,
        count(distinct po_num)  as number_po, 
        count(distinct cli_num) as number_cli
from    {existing_po}
group by  bought_for_before, bought_for_after,
        product_bought_before, product_bought_after,
        ape_cat_before, ape_cat_after,
        rider_type_before, rider_type_after,
        bought_hcr_for_before, bought_hcr_for_after, rider_hcr_before, rider_class_after,
        bought_mc_for_before, bought_mc_for_after,
        mth_since_lst_pur_cat
""",
existing_po = existing_po_pur_hist)

summary = existing_po_pur_sum.toPandas()

summary.to_csv(
    f"/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/existing_po_pur_hist.csv",
    index=False,
    header=True
)

In [0]:
existing_po_pur_hist.filter(
    F.col("total_ape_post_year").isNotNull()
).limit(10).display()

In [0]:
# Finding desire base (as of latest date)
inf_cus_base = spark.sql(f"""
with prod_list as (
select  PLAN_CODE, INS_TYP
        , case when fld.FLD_VALU_DESC='Investment' then
                case when pln.PLAN_CODE like 'RUV%' then 'Investment-ILP'
                    when pln.PLAN_CODE like 'UL%' then 'Investment-UL'
                    else 'Investment-Others' end
               when fld.FLD_VALU_DESC='Whole Life' then
                case when pln.PLAN_CODE not like '%ED99' then 'Endowment'
                    else 'Whole Life' end
               when pln.PLAN_CODE in ('FDB01','BIC01','BIC02','BIC03','BIC04') then 'Digital'
               when pln.PLAN_CODE in ('BHC9I','CA360','CI360','CN360','CX360','PN001') then 'Micro-Activator'
              else fld.FLD_VALU_DESC end
         as PROD_TYPE
from    vn_published_cas_db.tplans pln inner join
        vn_published_cas_db.tfield_values fld on pln.INS_TYP = fld.FLD_VALU and fld.FLD_NM='INS_TYP'
where   1=1
  and   pln.CVG_TYP = 'B'
), inf_cus as (
select    -- holding RUV
          distinct cpl.CLI_NUM as PO_NUM
          , collect_set(pln.PROD_TYPE) as LIST_OF_PRODUCT_TYPES
          , collect_set(rpl.PROD_TYP) as LIST_OF_RIDER_TYPES
          , cast(sum(pol.MODE_PREM*cast(PMT_MODE as int)/12)*1000/{ex_rate} as decimal(11,2)) as TOTAL_APE_USD
from      vn_published_casm_cas_snapshot_db.tpolicys pol 
    inner join
          vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on pol.POL_NUM = cpl.POL_NUM and cpl.LINK_TYP='O' and cpl.REC_STATUS='A' and pol.image_date = cpl.image_date
    inner join
          vn_published_casm_ams_snapshot_db.tams_agents agt on pol.AGT_CODE = agt.AGT_CODE and pol.image_date = agt.image_date
    left join
          vn_published_casm_cas_snapshot_db.tcoverages rid on pol.POL_NUM = rid.POL_NUM and rid.CVG_TYP='R' and rid.CVG_STAT_CD in ('1','2','3','5','7','9') and pol.image_date = rid.image_date
    left join 
          prod_list pln on pol.PLAN_CODE_BASE = pln.PLAN_CODE 
    left join
          vn_published_cas_db.tplans rpl on rid.PLAN_CODE = rpl.PLAN_CODE
where     1=1
    and   pol.image_date = '2025-03-31'
    and     pol.POL_STAT_CD in ('1','3','5')                            -- in-force or Maturing customer
    --and     agt.chnl_cd='01'                                          -- served by Agency channel
    --and     agt.comp_prvd_num in ('01','02','04','08','96','97','98')   -- Agency's compro (inc. orphan)
group by  cpl.CLI_NUM
)
select    a.*
from      inf_cus a
where     a.TOTAL_APE_USD > 0
""")

# Step 1: Create indicator columns for product types
result = inf_cus_base.withColumn(
    "ILP_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Investment-ILP"), "Y").otherwise("N")
).withColumn(
    "UL_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Investment-UL"), "Y").otherwise("N")
).withColumn(
    "ENDOWMENT_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Endowment"), "Y").otherwise("N")
).withColumn(
    "WHOLE_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Whole Life"), "Y").otherwise("N")
).withColumn(
    "TERM_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Term Life"), "Y").otherwise("N")
).withColumn(
    "DIGITAL_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Digital"), "Y").otherwise("N")
).withColumn(
    "MICRO_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Micro-Activator"), "Y").otherwise("N")
).withColumn(
    "OTH_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Accident"), "Y").otherwise("N")
)

# Step 2: Explode LIST_OF_RIDER_TYPES into dynamic columns
unique_rider_types = result.select(F.explode(F.col("LIST_OF_RIDER_TYPES")).alias("rider")).distinct().collect()
rider_type_columns = [rider.rider for rider in unique_rider_types]

for rider_type in rider_type_columns:
    result = result.withColumn(
        f"{rider_type}_IND",
        F.when(F.array_contains(F.col("LIST_OF_RIDER_TYPES"), rider_type), "Y").otherwise("N")
    )

result = result.drop('LIST_OF_PRODUCT_TYPES','LIST_OF_RIDER_TYPES')

In [0]:
# Add `rider_ind` and `product_type` flag
new_cols = ({
    'rider_ind': F.greatest(F.col('cancer_rider_ind'),
                            F.col('add_rider_ind'),
                            F.col('tpd_rider_ind'),
                            F.col('tp_rider_ind'),
                            F.col('mc_rider_ind'),
                            F.col('ci_rider_ind'),
                            F.col('hc_rider_ind')),

    'product_type': F.when((F.col('ilp_ind') == 'Y') &
                           (F.greatest(F.col('ul_ind'),
                                     F.col('endowment_ind'),
                                     F.col('whole_ind'),
                                     F.col('term_ind'),
                                     F.col('digital_ind'),
                                     F.col('micro_ind'),
                                     F.col('oth_ind')) == 'N'),
                           'ILP only')
                    .when((F.col('ul_ind') == 'Y') &
                           (F.greatest(F.col('ilp_ind'),
                                     F.col('endowment_ind'),
                                       F.col('whole_ind'),
                                       F.col('term_ind'),
                                       F.col('digital_ind'),
                                       F.col('micro_ind'),
                                       F.col('oth_ind')) == 'N'),
                           'UL only')
                    .when((F.col('endowment_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Endowment only')
                    .when((F.col('whole_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Whole Life only')
                    .when((F.col('term_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Term Life only')
                    .when((F.col('digital_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Digital only')
                    .when((F.col('micro_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Micro only')
                    .when((F.col('oth_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                      F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind')) == 'N'),
                          'Accident only')
                    .when(F.greatest(F.col('ilp_ind'),
                                    F.col('ul_ind'),
                                    F.col('endowment_ind'),
                                    F.col('whole_ind'),
                                    F.col('term_ind'),
                                    F.col('digital_ind'),
                                    F.col('micro_ind'),
                                    F.col('oth_ind')) == 'Y', 'Multiple products')
})

result = result.withColumns(new_cols)
result = result.toDF(*[col.lower() for col in result.columns])

In [0]:
# Select only unique PO
existing_po_df = spark.sql("""
select  po_num, pol_num, SELLER as agt_code,
        sbmt_dt, issue_date,
        AGE_LA_cat, BASE_APE_cat, RIDER_APE_cat, TOTAL_CLAIM_AMOUNT_cat, CLAIM_AMOUNT_PER_YEAR_cat,
        row_number() over (partition by PO_NUM order by ISSUE_DATE desc, POL_NUM desc) as rn
from    {existing_po}
where   EXISTING_PO_IND = 'Y'
qualify rn = 1               
""",
existing_po = df
).drop("rn")

# Derive repurchase HP scores
onboard_path = f"/mnt/prod/Curated/VN/Master/VN_CURATED_CUSTOMER_ANALYTICS_DB/ONBOARDED_CUSTOMER_SCORE/image_date={lst_mthend}"
existing_path = f"/mnt/prod/Curated/VN/Master/VN_CURATED_CUSTOMER_ANALYTICS_DB/EXISTING_CUSTOMER_SCORE/image_date={lst_mthend}"

new_leads_score = spark.read.format("parquet").load(onboard_path).select("po_num", "decile").orderBy("po_num", "decile")
existing_leads_score = spark.read.format("parquet").load(existing_path).select("po_num", "decile").orderBy("po_num", "decile")
leads_score = new_leads_score.union(existing_leads_score
).dropDuplicates(["po_num"]
).fillna({"decile": 10})

leads_score = leads_score.withColumn(
  "decile_cat",
  F.when(F.col("decile") <= 2, "1 Top1-2-HP")
  .when(F.col("decile") <= 3, "2 Top3-HP")
  .when(F.col("decile") <= 5, "3 Top4-5-HP")
  .otherwise("4 Bottom6-10-LP")
).orderBy("decile_cat")

# Load latest Agency Structure from MIS
loc_path = f"/mnt/lab/vn/project/scratch/adhoc/loc_to_psm_mapping_202504.xlsx"
sheet_name = "Structure"
cell_pos = "A1"
loc_to_psm = spark.read.format(
    "com.crealytics.spark.excel"
).option(
    "dataAddress", f"{sheet_name}!{cell_pos}"
).option(
    "header", "true"
).option(
    "treatEmptyValuesAsNulls", "true"
).option(
    "inferSchema", "true"
).option(
    "addColorColumns", "False"
).load(
    loc_path
)

# Derive Agent status and Group
mpro_title = spark.sql("""
SELECT  agt.AGT_CODE AS agt_code,
        CASE WHEN stat_cd = '01' THEN 'INFORCE'
             WHEN stat_cd = '99' THEN 'TERMINATED'
        END AS agt_status,
        CASE WHEN trmn_dt IS NOT NULL THEN '4.Terminated'
             WHEN trmn_dt IS NULL THEN
                 CASE
                     WHEN comp_prvd_num IN ('01','02','08','96') THEN '1.Inforce'
                     WHEN comp_prvd_num = '04' THEN '2.CSC'
                     WHEN comp_prvd_num IN ('97','98') THEN '3.SM'
                     ELSE '5.Not-Agency'
                 END
        END AS AGT_RLTNSHP,
        COALESCE(
            CASE WHEN agt.MPRO_TITLE IS NOT NULL THEN
                CASE WHEN MDRT_DESC IN ('TOT', 'COT') THEN 'MDRT'
                     ELSE MDRT_DESC
                END
                ELSE 'Normal'
            END,
            MPRO_TITLE
        ) AS mpro_title,
        lagt.loc_cd,
        NVL(lagt.psm_name_8, 'Open') AS psm_name_8,
        NVL(lagt.psm_name_9, 'Open') AS psm_name_9,
        agt.image_date
FROM    vn_published_casm_ams_snapshot_db.tams_agents agt
    LEFT JOIN 
        {loc_to_sm_mapping} lagt ON agt.LOC_CODE = lagt.loc_cd
WHERE   1=1
    AND agt.image_date between '2024-01-31' and '2025-05-31'
    AND agt.image_date <> '2025-04-30'
    --AND MPRO_TITLE in ('P','G','S')
    --and to_date(MPRO_TITLE_EFF_DT) between '2024-04-01' and '2025-02-28'
    --AND TRMN_DT IS NULL
""",
loc_to_sm_mapping = loc_to_psm
)

par1_df = spark.read.format("parquet").load(
    f"/mnt/lab/vn/project/cpm/datamarts/TPARDM_MTHEND/"
).filter(
    (F.col("image_date") >= F.lit("2024-01-31")) &
    (F.col("image_date") <= F.lit("2025-03-31"))
).select(
    "agt_cd", "last_9m_pol", "last_3m_pol", "last_mth_pol", "image_date"
).withColumnRenamed(
    "agt_cd", "agt_code"
).dropDuplicates()

# Derive total_cus before the issue month
par3_df = spark.read.format("parquet").load(
    f"/mnt/lab/vn/project/cpm/datamarts/TPARDM_MTHEND/"
).filter(
    (F.col("image_date") >= F.lit("2023-12-31")) &
    (F.col("image_date") <= F.lit("2025-02-28"))
).select(
    "agt_cd", "total_cus", "image_date"
).withColumnRenamed(
    "agt_cd", "agt_code"
).withColumn(
    "image_date",
    F.last_day(F.date_add(F.col("image_date"), 1))
).dropDuplicates()

par1_df = par1_df.join(
    par3_df,
    on=["agt_code", "image_date"]
).select(
    "agt_code", "last_9m_pol", "last_3m_pol", "last_mth_pol", "total_cus", "image_date"
)

# Agent tier and status as of latest month
par2_df = spark.read.format("parquet").load(
    f"/mnt/lab/vn/project/cpm/datamarts/TPARDM_MTHEND/"
).filter(
    (F.col("image_date") == F.lit("2025-05-31"))
).select(
    "agt_cd", "last_9m_pol", "last_3m_pol", "last_mth_pol", "total_cus", "image_date"
).withColumnRenamed(
    "agt_cd", "agt_code"
).dropDuplicates()

par_df = par1_df.union(par2_df)

mpro_title = mpro_title.join(
    par_df, 
    on=["agt_code", "image_date"],
    how="left"
)

mpro_title = mpro_title.withColumn(
    "agt_actvness",
    F.when(F.col("last_mth_pol") > 0, "1m active")
    .when(F.col("last_3m_pol") > 0, "3m active")
    .when(F.col("last_9m_pol") > 0, "9m active")
    .otherwise("SA")
).select(
    "agt_code", "loc_cd", "mpro_title", "psm_name_8", "psm_name_9", "agt_rltnshp", "agt_actvness", "total_cus", "image_date"
)

mpro_title = mpro_title.withColumn(
    "agt_segment",
    F.when(F.col("agt_actvness") != "SA",
        F.when(F.col("mpro_title") != "Normal", F.col("mpro_title"))
        .otherwise(F.col("agt_actvness"))
    ).otherwise(
        F.when(F.col("agt_rltnshp").isin("2.CSC","3.SM","4.Terminated"), "UCM")
        .otherwise("SA")
    )
)

#mpro_title.createOrReplaceTempView("mpro_title")

# Derive PO age, marital status, tenure, policy holding, time from last purchase
existing_po_profile = spark.sql("""
with po_age_dtl as (
select  nb.po_num,
        floor(months_between(cus.image_date, to_date(cus.BIRTH_DT)) / 12) as cus_current_age,
        cus.image_date
from    {nb_pol_list} nb
    inner join
        vn_published_casm_cas_snapshot_db.tclient_details cus on nb.po_num = cus.cli_num
where   cus.image_date= '2025-03-31'
), po_age as (
select  *,
        case when cus_current_age < 25 then '1. <25yo'
             when cus_current_age between 25 and 29 then '2. 25 - 29yo'
             when cus_current_age between 30 and 34 then '3. 30 - 34yo'
             when cus_current_age between 35 and 39 then '4. 35 - 39yo'
             when cus_current_age between 40 and 44 then '5. 40 - 44yo'
             when cus_current_age between 45 and 49 then '6. 45 - 49yo'
             when cus_current_age between 50 and 54 then '7. 50 - 54yo'
             else '8. >=55yo'
        end as po_age_cat
from    po_age_dtl
), frst_bought_by_existing_po as (
select  cpl.cli_num as po_num,
        to_date(min(sbmt_dt)) as frst_sbmt_dt_by_existing_po
from    {nb_pol_list} nb
inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on 
        (nb.pol_num = cpl.pol_num and cpl.link_typ = 'O' and cpl.rec_status = 'A')
where   cpl.image_date = '2025-03-31'
group by
        cpl.cli_num
), last_pur_pols as (
select  nb.po_num, months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) as months_since_last_purchase,
        case 
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 6 then  '1. < 6m'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 12 then '2. < 1yr'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 36 then '3. 1 - 2yr'
          when months_between(nb.frst_sbmt_dt_by_existing_po, pol.pol_eff_dt) < 60 then '4. 3 - 5yr'
          else '5. >5yr'
        end as mth_since_lst_pur_cat,
        pol.pol_num, to_date(pol.pol_eff_dt) as pol_eff_dt,
        case when pol.plan_code_base in ('UL004','UL005','UL035','UL036') then 'UL-Single-Premium'
              when pol.plan_code_base not in ('UL004','UL005','UL035','UL036') and pol.plan_code_base like 'UL%' then 'ULI'
              when pol.plan_code_base like 'RUV%' then 'UILP'
              when pol.plan_code_base like 'RP%' then 'TROP'
              when pol.plan_code_base in ('BHC9I','PN001','PA007','PA008') then 'Single-Year'
              when pol.plan_code_base in ('FDB01','BIC01','BIC02','BIC03','BIC04') then 'Digital'
              when pol.plan_code_base in ('CA360','CI360','CN360','CX360','MI007') then 'Activator-Micro'
              else 'Traditional'
        end as product_type,
        row_number() over (partition by nb.po_num order by pol.pol_eff_dt desc, pol.pol_num desc) rn
from    vn_published_casm_cas_snapshot_db.tpolicys pol
inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on 
        (pol.pol_num = cpl.pol_num and pol.image_date = cpl.image_date and cpl.link_typ = 'O' and cpl.rec_status = 'A')
inner join
        frst_bought_by_existing_po nb on (cpl.cli_num = nb.po_num and pol.pol_eff_dt < nb.frst_sbmt_dt_by_existing_po)
where   1 = 1
  and   pol.pol_stat_cd not in ('A', 'N', 'R', 'X')
  and   pol.image_date = '2025-03-31'
qualify rn = 1
), frst_eff_pol as (
select  cpl.image_date, cpl.cli_num as po_num,
        to_date(min(pol_eff_dt)) as frst_eff_dt
from    vn_published_casm_cas_snapshot_db.tpolicys pol
    inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on
        (pol.image_date = cpl.image_date and pol.pol_num = cpl.pol_num and cpl.link_typ = 'O' and cpl.rec_status = 'A')
where   cpl.image_date = '2025-03-31'
    and pol.pol_stat_cd not in ('A','N','R','X')
group by cpl.image_date, cpl.cli_num
), po_tenure as (
select  *,
        case when months_between(image_date, frst_eff_dt) < 6 then '1. < 6m'
             when months_between(image_date, frst_eff_dt) < 12 then '2. < 1yr'
             when months_between(image_date, frst_eff_dt) < 24 then '3. 1 - 2yr'
             when months_between(image_date, frst_eff_dt) < 36 then '4. 2 - 3yr'
             when months_between(image_date, frst_eff_dt) < 60 then '5. 3 - 5yr'
             when months_between(image_date, frst_eff_dt) < 84 then '6. 5 - 7yr'
             when months_between(image_date, frst_eff_dt) < 120 then '7. 7 - 10yr'
             else '8. >10yr'
        end as po_tenure_cat
from    frst_eff_pol
), po_pols as (
select   cpl.cli_num as po_num, count(distinct pol.pol_num) as pol_count, cpl.image_date
from     vn_published_casm_cas_snapshot_db.tclient_policy_links cpl
    inner join
        vn_published_casm_cas_snapshot_db.tpolicys pol on (cpl.pol_num = pol.pol_num and cpl.link_typ = 'O' and cpl.rec_status = 'A'
                                                        and pol.image_date = cpl.image_date)
where   cpl.image_date = '2025-03-31'
    and pol.pol_stat_cd in ('1', '3', '5')
group by
        cpl.cli_num, cpl.image_date
), po_holding as (
select  *,
        case when pol_count <= 1 then '1. 1 policy'
             when pol_count <= 2 then '2. 2 policies'
             when pol_count <= 5 then '3. 3 - 5 policies'
             else '4. >5 policies'
        end as po_holding_cat
from    po_pols
)/*, agt_po as (
select  agt_code, count(distinct cpl.cli_num) as total_cus
from    vn_published_casm_cas_snapshot_db.tpolicys pol
    inner join
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on (cpl.pol_num = pol.pol_num 
        and cpl.link_typ = 'O' 
        and cpl.rec_status = 'A'
        and pol.image_date = cpl.image_date)
where   pol.image_date = '2025-03-31'
group by
        agt_code
)*/
select  nb.po_num, nb.agt_code, sts.total_cus, mpo.total_cus as total_cus_before,
        nvl(age.po_age_cat, '9. Unknown') as po_age_cat,
        nvl(ten.po_tenure_cat, '9. Unknown') as po_tenure_cat,
        nvl(hld.po_holding_cat, '5. Unknown') as po_holding_cat,
        nvl(pur.mth_since_lst_pur_cat, '6. Unknonw') as mth_since_lst_pur_cat,
        nvl(ilp_ind, 'N') as ilp_ind,
        nvl(ul_ind, 'N') as ul_ind,
        nvl(endowment_ind, 'N') as endowment_ind,
        nvl(whole_ind, 'N') as whole_ind,
        nvl(term_ind, 'N') as term_ind,
        nvl(digital_ind, 'N') as digital_ind,
        nvl(micro_ind, 'N') as micro_ind,
        nvl(oth_ind, 'N') as oth_ind,
        nvl(cancer_rider_ind, 'N') as cancer_rider_ind,
        nvl(mc_rider_ind, 'N') as mc_rider_ind,
        nvl(ci_rider_ind, 'N') as ci_rider_ind,
        nvl(tpd_rider_ind, 'N') as tpd_rider_ind,
        nvl(tp_rider_ind, 'N') as tp_rider_ind,
        nvl(add_rider_ind, 'N') as add_rider_ind,
        nvl(hc_rider_ind, 'N') as hc_rider_ind,
        nvl(por.product_type, 'None') as product_type,
        nvl(sco.decile_cat, 'Missing') as decile_cat,
        nvl(mpo.agt_segment, 'Other') as agt_segment,
        nvl(sts.agt_segment, 'Other') as agt_latest_segment,
        nvl(mpo.psm_name_8, 'Open') as psm_name_8,
        nvl(mpo.psm_name_9, 'Open') as psm_name_9,
        nvl(vip.VIP_TYP_DESC, 'NonVIP') as vip_status
from    {nb_pol_list} nb
    --left join
    --    agt_po po on nb.agt_code = po.agt_code
    left join
        po_age age on nb.po_num = age.po_num
    left join
        po_tenure ten on nb.po_num = ten.po_num
    left join
        po_holding hld on nb.po_num = hld.po_num
    left join
        last_pur_pols pur on nb.po_num = pur.po_num
    left join
        {leads_score} sco on nb.po_num = sco.po_num
    left join
        {mpro_title} mpo on nb.agt_code = mpo.agt_code and last_day(nb.issue_date) = mpo.image_date
    left join
        {mpro_title} sts on nb.agt_code = sts.agt_code and sts.image_date = '2025-03-31'
    left join
        {portfolio} por on nb.po_num = por.po_num
    left join
        vn_published_casm_cas_snapshot_db.twrk_client_ape vip on nb.po_num = vip.cli_num and vip.image_date = '2025-03-31'
""",
nb_pol_list = existing_po_df,
leads_score = leads_score,
mpro_title = mpro_title,
portfolio = result)

# record_count = existing_po_profile.count()
# po_count = existing_po_profile.select("po_num").distinct().count()
# print(record_count, po_count)
# check_dup(existing_po_profile, "po_num")

In [0]:
summary = existing_po_profile.toPandas()
summary.to_csv(
    f"/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/existing_po_profile.csv",
    index=False,
    header=True
)
summary.head(2)

In [0]:
# mpro_title.groupBy(
#     "image_date",
#     "agt_segment"
# ).agg(
#     F.count("agt_code").alias("agt_count"),
#     F.sum("total_cus").alias("serv_cus_count")
# ).display()

### DEEP-DIVE ANALYSIS ON EXISTING LA COHORT

In [0]:
# claims_df = spark.sql(f"""
# -- retrieve claim payment dates
# WITH results_dedup AS (
#   SELECT DISTINCT
#          POL_NUM,
#          CLI_NUM,
#          RULE_ID,
#          err_en_msg, TYPE, ASSIGN_TEAM, trxn_dt, rerun_seq
#          ,DENSE_RANK() OVER (PARTITION BY POL_NUM ORDER BY rerun_seq DESC) AS rank
#   FROM   vn_published_cas_db.tpolicy_results
#   QUALIFY rank = 1
# ), order_results AS (
#   SELECT POL_NUM,
#          CLI_NUM,
#          RULE_ID,
#          err_en_msg,
#          CASE 
#            WHEN TYPE = 'C' THEN 'C-Client'
#            WHEN TYPE = 'P' THEN 'P-Policy'
#            WHEN TYPE = 'A' THEN 'A-Agent'
#            ELSE TYPE
#          END AS TYPE,
#          ASSIGN_TEAM,
#          ROW_NUMBER() OVER(PARTITION BY POL_NUM ORDER BY POL_NUM) AS rn
#   FROM   results_dedup
# ),
# sorted_remarks AS (
#   SELECT POL_NUM, CLI_NUM, RULE_ID, TYPE, ASSIGN_TEAM, err_en_msg
#   FROM order_results
# ),
# clm_his AS (
# SELECT POL_NUM, CLI_NUM,
#        CONCAT_WS('/', COLLECT_SET(TYPE)) AS TYPES,
#        CONCAT_WS('>', COLLECT_LIST(err_en_msg)) AS ADMIN_REMARKS
# FROM sorted_remarks
# WHERE   TYPE='C-Client'
#   AND   err_en_msg like '%Client has claim history%'
# GROUP BY POL_NUM, CLI_NUM
# ), 
# clm_type as (
# select  cast(fld_valu as int) AS clm_code
#         , fld_valu_desc AS benefit
#         ,case when cast(fld_valu as int) in (3,7,11,31,36,50,51) then 'Quyền lợi trợ cấp-Medicash'
#           when cast(fld_valu as int) in (27,28,29,40,41,42) then 'Quyền lợi điều trị-Healthcare'
#           when cast(fld_valu as int) in (9,38,45) then 'Quyền lượi thai sản-Female Benefits' 
#           else 'Quyền lợi lớn khác'
#      end AS clm_benefit_type
# from    vn_published_cas_db.tfield_values
# where   fld_nm='CLM_CODE'
# ), 
# t_dnr_sub_a as (
# 	select
# 		payee
# 		,max(pmt_dt) pmt_dt
# 	from
# 		vn_published_cas_db.tdnr_details
# 	where
# 		dnr_stat_cd <> 'X'
# 		and pmt_stat_cd not in ('N','W')
# 		and payo_reas in ('912','913','914','915','916','917','918','919','920','921','927','928','929','930')							
# 	group by
# 		payee
# ),
# t_dnr_sub_b as (
# 	select
# 		payee
# 		,pmt_dt
# 		,max(payo_type) payo_type
# 	from
# 		vn_published_cas_db.tdnr_details
# 	where
# 		dnr_stat_cd <> 'X'
# 		and pmt_stat_cd not in ('N','W')
# 		and payo_reas in ('912','913','914','915','916','917','918','919','920','921','927','928','929','930')							
# 	group by
# 		payee
# 		,pmt_dt
# ),
# t_dnr AS (
# select
# 	a.payee
# 	,case
# 		when c.direct_billing = 'Y' then c.clm_recv_dt
# 		when c.payo_typ = 'ANM' then c.clm_aprov_dt
# 		when c.payo_typ is null and c.case_owner in ('INSMARTHCM','INSMARTHN') then date_add(c.clm_aprov_dt,3)
# 		when c.payo_typ in ('TF','PO') then date_add(c.clm_aprov_dt,3)
# 		when a.pmt_dt is null then c.clm_aprov_dt
# 		else a.pmt_dt
# 	end pmt_dt
# 	,b.payo_type
# from	
# 	t_dnr_sub_a a
# 	inner join t_dnr_sub_b b on	(a.payee = b.payee and a.pmt_dt = b.pmt_dt)
#     inner join vn_published_cas_db.tclaim_details c on a.payee = c.clm_id
# ),
# all_claims as (
# SELECT	clm_id, pol_num, cli_num, clm_code, clm_eff_dt, clm_aprov_dt, clm_aprov_amt
# FROM   	vn_published_cas_db.tclaim_details
# UNION
# SELECT	clm_id, pol_num, con.CLI_CNSLDT_NUM AS cli_num, clm_code, clm_eff_dt, clm_aprov_dt, clm_aprov_amt
# FROM   	vn_published_cas_db.tclaim_details clm
#   INNER JOIN
#         vn_published_cas_db.tclient_consolidations con ON clm.CLI_NUM = con.CLI_NUM
# )
# SELECT DISTINCT
#     clm.clm_id
#     ,clm.pol_num
#     ,nb.cli_num
# 	,SUBSTR(TO_DATE(clm_eff_dt),1,7) AS clm_mth
#     ,SUBSTR(TO_DATE(t_dnr.pmt_dt),1,7) AS pmt_mth
#     ,CAST(NVL(clm.clm_aprov_amt,0) AS INT) AS clm_amt
#     ,typ.clm_benefit_type
#     ,CASE WHEN clm.clm_eff_dt > nb.issue_date THEN 0 ELSE 1 END AS valid_his_clm_ind
# FROM all_claims clm LEFT JOIN
#      clm_his his ON clm.cli_num = his.cli_num LEFT JOIN
#      nb_pol_list nb on his.POL_NUM = nb.POL_NUM LEFT JOIN
#      t_dnr ON clm.clm_id=t_dnr.payee LEFT JOIN
#      clm_type typ ON clm.clm_code=typ.clm_code
# WHERE 1=1
# """)

# clm_sum_df = claims_df.groupBy('cli_num') \
#     .agg(
#         F.count('clm_id').alias('total_claims'),
#         F.sum('clm_amt').alias('total_claim_amount'),
#         F.concat_ws('/',F.collect_set(F.col('clm_benefit_type'))).alias('claim_benefits'),
#         F.concat_ws('/',F.collect_set(F.col('clm_mth'))).alias('claim_months'),
#         F.count(F.when(F.col('clm_benefit_type') == 'Quyền lợi trợ cấp-Medicash', F.col('pol_num'))).alias('medicash_claims'),
#         F.sum(F.when(F.col('clm_benefit_type') == 'Quyền lợi trợ cấp-Medicash', F.col('clm_amt'))).alias('medicash_claim_amount'),
#         F.count(F.when(F.col('clm_benefit_type') == 'Quyền lợi điều trị-Healthcare', F.col('pol_num'))).alias('healthcare_claims'),
#         F.sum(F.when(F.col('clm_benefit_type') == 'Quyền lợi điều trị-Healthcare', F.col('clm_amt'))).alias('healthcare_claim_amount'),
#         F.count(F.when(F.col('clm_benefit_type') == 'Quyền lợi thai sản-Female Benefits', F.col('pol_num'))).alias('female_claims'),
#         F.sum(F.when(F.col('clm_benefit_type') == 'Quyền lợi thai sản-Female Benefits', F.col('clm_amt'))).alias('female_claim_amount'),
#         F.count(F.when(F.col('clm_benefit_type') == 'Quyền lợi lớn khác', F.col('pol_num'))).alias('other_claims'),
#         F.sum(F.when(F.col('clm_benefit_type') == 'Quyền lợi lớn khác', F.col('clm_amt'))).alias('other_claim_amount'),
#         F.max(F.col("valid_his_clm_ind")).alias("valid_his_clm_ind")
#     )

# clm_sum_df.createOrReplaceTempView("claims")

In [0]:
existing_la_df = spark.sql(f"""
SELECT  nb.*,        
        med_his_ind, excl_ind, rating_his_ind, occp_cls_ind, decl_ind
FROM    nb_pol_list nb
    LEFT JOIN
        vn_aa_reports.pre_approved_exclusion exc ON nb.CLI_NUM = exc.cli_num
WHERE   1=1
    AND nb.ais_decision <> 'Clean'
    AND nb.admin_rule_category = 'Client - Claim/rating/exclusion/rejected/rescind'
    --AND nb.EXISTING_LA_IND = 'Y'
    AND nb.claim_benefits IS NOT NULL
""").drop(
    "stp_flag", "hit_rule"
).fillna(
    {"claim_benefits": "None"}
)

existing_la_count = existing_la_df.count()
print("Total Existing LA count:", existing_la_count)

In [0]:
existing_la_df.toPandas().to_csv("/dbfs/mnt/lab/vn/project/VN_NB/CSV_FILES/VN_NB_existing_la_rawdata_v3.csv", index=False, header=True, encoding='utf-8-sig')